# Baby Step 6 - Constrained Structure Optimization

Evaluate alternative structures with transparent economic and legal assumptions, hard governance constraints, and four stress scenarios per alternative.

**Synthetic-data boundary:** no filing, communication, appointment, restructuring, or real-world tax action is authorized.

## Deterministic engine

The following cell contains the complete implementation, including data transformations, audit files, vault notes, validation, manifests, and packaging logic.

In [ ]:
# Self-contained implementation for Baby Step 6
from __future__ import annotations

import csv
import hashlib
import html
import inspect
import json
import math
import shutil
import textwrap
import zipfile
from collections import Counter, defaultdict
from datetime import date
from pathlib import Path


ROOT = Path(__file__).resolve().parent
BUILD_DATE = date(2026, 7, 21).isoformat()
QUARTER = "2026-Q3"
NEXT_QUARTER = "2026-Q4"


def read_csv(path: Path) -> list[dict]:
    with path.open(encoding="utf-8", newline="") as stream:
        return list(csv.DictReader(stream))


def write_csv(path: Path, rows: list[dict], fieldnames: list[str] | None = None) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    if not rows and not fieldnames:
        raise ValueError(f"Cannot infer fields for empty CSV: {path}")
    names = fieldnames or list(rows[0])
    with path.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=names, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)


def write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text.rstrip() + "\n", encoding="utf-8")


def write_json(path: Path, value) -> None:
    write_text(path, json.dumps(value, indent=2, ensure_ascii=False))


def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def md_table(rows: list[dict], columns: list[tuple[str, str]]) -> str:
    header = "| " + " | ".join(label for _, label in columns) + " |"
    rule = "|" + "|".join("---" for _ in columns) + "|"
    body = ["| " + " | ".join(str(row.get(key, "")) for key, _ in columns) + " |" for row in rows]
    return "\n".join([header, rule, *body])


def copy_vault(step: int) -> tuple[Path, Path, Path]:
    source = ROOT / f"Tax_Planning_ExoBrain_Step_{step - 1}" / "Tax_Planning_ExoBrain_Vault"
    package = ROOT / f"Tax_Planning_ExoBrain_Step_{step}"
    target = package / "Tax_Planning_ExoBrain_Vault"
    if not source.exists():
        raise FileNotFoundError(f"Missing prior-step vault: {source}")
    if package.exists():
        shutil.rmtree(package)
    shutil.copytree(source, target)
    return source, package, target


def zip_tree(source: Path, destination: Path, root_name: str | None = None) -> None:
    if destination.exists():
        destination.unlink()
    with zipfile.ZipFile(destination, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=9) as archive:
        for file in sorted(p for p in source.rglob("*") if p.is_file()):
            rel = file.relative_to(source)
            arc = Path(root_name) / rel if root_name else rel
            archive.write(file, arc.as_posix())


def csv_count(data: Path, name: str) -> int:
    return len(read_csv(data / name))


def append_recommendation_note(vault: Path, rec_id: str, heading: str, body: str) -> None:
    path = vault / "09_Recommendations" / f"{rec_id}.md"
    if path.exists():
        write_text(path, path.read_text(encoding="utf-8") + f"\n\n## {heading}\n\n{body}")


def create_notebook(step: int, package: Path, title: str, objective: str, expected: str) -> Path:
    script = Path(__file__).read_text(encoding="utf-8")
    script = script.replace("if __name__ == \"__main__\":\n    main()", "")
    code = f"""# Self-contained implementation for Baby Step {step}\n{script}\n"""
    run = f"""from google.colab import drive
drive.mount('/content/drive')

# The notebook builds the prior stages first in a temporary Colab workspace,
# then writes the requested cumulative vault and package to Drive.
ROOT = Path('/content/tax_planning_exobrain_build')
ROOT.mkdir(parents=True, exist_ok=True)
PROJECT = Path('/content/drive/MyDrive/Tax_Planning_ExoBrain_Project')

# Place the prior step package folder in ROOT, or change ROOT to the project
# directory that already contains Tax_Planning_ExoBrain_Step_{step - 1}.
prior = PROJECT / 'Step_{step - 1}' / 'Tax_Planning_ExoBrain_Step_{step - 1}'
if prior.exists() and not (ROOT / prior.name).exists():
    shutil.copytree(prior, ROOT / prior.name)

build_one({step})
out = ROOT / 'Tax_Planning_ExoBrain_Step_{step}'
destination = PROJECT / 'Step_{step}'
if destination.exists():
    shutil.rmtree(destination)
shutil.copytree(out, destination / out.name)
print(json.loads((out / 'STEP_{step}_VALIDATION.json').read_text())['status'])
print(destination)
"""
    nb = {
        "nbformat": 4,
        "nbformat_minor": 5,
        "metadata": {"colab": {"name": f"Baby Step {step} - Tax Planning Exo-Brain"}, "kernelspec": {"name": "python3", "display_name": "Python 3"}},
        "cells": [
            {"cell_type": "markdown", "metadata": {}, "source": [f"# Baby Step {step} - {title}\n\n{objective}\n\n**Synthetic-data boundary:** no filing, communication, appointment, restructuring, or real-world tax action is authorized."]},
            {"cell_type": "markdown", "metadata": {}, "source": ["## Deterministic engine\n\nThe following cell contains the complete implementation, including data transformations, audit files, vault notes, validation, manifests, and packaging logic."]},
            {"cell_type": "code", "execution_count": None, "metadata": {}, "outputs": [], "source": code.splitlines(keepends=True)},
            {"cell_type": "markdown", "metadata": {}, "source": ["## Execute in Google Drive\n\nThe prior cumulative package is treated as immutable input. The result is written to a new step folder."]},
            {"cell_type": "code", "execution_count": None, "metadata": {}, "outputs": [], "source": run.splitlines(keepends=True)},
            {"cell_type": "markdown", "metadata": {}, "source": [f"## Expected validated result\n\n{expected}"]},
        ],
    }
    path = package / f"Baby_Step_{step}_Tax_Planning_ExoBrain.ipynb"
    write_json(path, nb)
    return path


def finalize_step(step: int, package: Path, validation: dict, principal_files: list[str]) -> None:
    validation["step"] = step
    validation["date"] = BUILD_DATE
    validation["status"] = "PASS" if all(validation["checks"].values()) else "FAIL"
    write_json(package / f"STEP_{step}_VALIDATION.json", validation)

    manifest_rows = []
    for file in sorted(p for p in package.rglob("*") if p.is_file() and p.name not in {"SHA256SUMS.txt", f"STEP_{step}_MANIFEST.csv"}):
        manifest_rows.append({
            "relative_path": file.relative_to(package).as_posix(),
            "bytes": file.stat().st_size,
            "sha256": sha256(file),
            "artifact_class": "vault" if "Tax_Planning_ExoBrain_Vault" in file.parts else "deliverable",
            "step": step,
        })
    write_csv(package / f"STEP_{step}_MANIFEST.csv", manifest_rows)
    checksum_lines = [f"{row['sha256']}  {row['relative_path']}" for row in manifest_rows]
    checksum_lines.append(f"{sha256(package / f'STEP_{step}_MANIFEST.csv')}  STEP_{step}_MANIFEST.csv")
    write_text(package / "SHA256SUMS.txt", "\n".join(checksum_lines))

    current_vault = ROOT / f"Tax_Planning_ExoBrain_Current_Vault_Step_{step}.zip"
    zip_tree(package / "Tax_Planning_ExoBrain_Vault", current_vault, "Tax_Planning_ExoBrain_Vault")
    step_zip = ROOT / f"Tax_Planning_ExoBrain_Step_{step}.zip"
    zip_tree(package, step_zip, package.name)

    download_dir = ROOT / f"Tax_Planning_ExoBrain_Step_{step}_Downloadables"
    if download_dir.exists():
        shutil.rmtree(download_dir)
    download_dir.mkdir()
    for name in principal_files + [f"Baby_Step_{step}_Tax_Planning_ExoBrain.ipynb", f"STEP_{step}_VALIDATION.json", f"STEP_{step}_MANIFEST.csv", "SHA256SUMS.txt"]:
        source = package / name
        if source.exists():
            shutil.copy2(source, download_dir / source.name)
    shutil.copy2(current_vault, download_dir)
    shutil.copy2(step_zip, download_dir)
    zip_tree(download_dir, ROOT / f"Tax_Planning_ExoBrain_Step_{step}_Downloadables.zip", download_dir.name)

    with zipfile.ZipFile(step_zip) as archive:
        bad = archive.testzip()
        if bad:
            raise AssertionError(f"Corrupt member in {step_zip}: {bad}")


def build_step5() -> None:
    _, package, vault = copy_vault(5)
    data = vault / "16_Data"
    conditions = read_csv(data / "committee_conditions.csv")
    decisions = read_csv(data / "committee_decisions.csv")
    recs = read_csv(data / "recommendations.csv")
    claims = read_csv(data / "atomic_tax_claims.csv")
    contradictions = read_csv(data / "contradictions.csv")
    rec_by_id = {r["recommendation_id"]: r for r in recs}
    decision_by_rec = {d["recommendation_id"]: d for d in decisions}

    evidence_rows, finding_rows = [], []
    evidence_types = ["DOCUMENT", "INTERVIEW_NOTE", "CALCULATION_REPERFORMANCE"]
    for i, condition in enumerate(conditions, 1):
        rec_id = condition["recommendation_id"]
        excluded = decision_by_rec[rec_id]["diligence_tier"] == "EXCLUDED"
        if excluded:
            disposition, finding = "REDESIGN_REQUIRED", "Evidence cannot cure the substance-led design defect; return to operating-model redesign."
        elif (i + int(rec_id[4:7])) % 7 == 0:
            disposition, finding = "OPEN_EXCEPTION", "Evidence is directionally supportive but an exception owner and expiry date remain required."
        else:
            disposition, finding = "CLOSED_SYNTHETIC", "Three-source synthetic triangulation satisfies the bounded diligence standard."
        evidence_ids = []
        for sequence, evidence_type in enumerate(evidence_types, 1):
            ev_id = f"EVD5-{i:03d}-{sequence}"
            evidence_ids.append(ev_id)
            quality = 48 if excluded else (68 + ((i * 7 + sequence * 5) % 28))
            evidence_rows.append({
                "evidence_id": ev_id, "condition_id": condition["condition_id"], "recommendation_id": rec_id,
                "conglomerate_id": condition["conglomerate_id"], "evidence_type": evidence_type,
                "evidence_description": f"Synthetic {evidence_type.lower().replace('_', ' ')} for {condition['condition_type']}.",
                "source_owner_role": "Synthetic Business Owner" if sequence == 1 else "Independent Synthetic Reviewer",
                "as_of": BUILD_DATE, "quality_score": quality, "independence_checked": True,
                "supports_closure": disposition == "CLOSED_SYNTHETIC", "content_hash": hashlib.sha256(f"{ev_id}|{quality}|{disposition}".encode()).hexdigest(),
                "transmitted": False, "synthetic": True,
            })
        finding_rows.append({
            "finding_id": f"FND5-{i:03d}", "condition_id": condition["condition_id"], "recommendation_id": rec_id,
            "conglomerate_id": condition["conglomerate_id"], "condition_type": condition["condition_type"],
            "evidence_ids": ";".join(evidence_ids), "evidence_count": 3, "disposition": disposition,
            "finding": finding, "reviewer_role": "Synthetic Diligence Reviewer", "human_review_required": True,
            "implementation_authorized": False, "synthetic": True,
        })
        condition["status"] = disposition
        condition["step5_finding_id"] = f"FND5-{i:03d}"

    findings_by_rec: defaultdict[str, list[dict]] = defaultdict(list)
    for row in finding_rows:
        findings_by_rec[row["recommendation_id"]].append(row)
    summary_rows = []
    for rec in recs:
        rows = findings_by_rec[rec["recommendation_id"]]
        counts = Counter(r["disposition"] for r in rows)
        if counts["REDESIGN_REQUIRED"]:
            outcome, confidence = "REMAIN_ON_HOLD", 48
        elif counts["OPEN_EXCEPTION"]:
            outcome, confidence = "DILIGENCE_COMPLETE_WITH_EXCEPTIONS", 72 + int(rec["recommendation_id"][4:7])
        else:
            outcome, confidence = "DILIGENCE_COMPLETE_BOUNDED", 84 + int(rec["recommendation_id"][4:7])
        summary_rows.append({
            "recommendation_id": rec["recommendation_id"], "conglomerate_id": rec["conglomerate_id"], "conglomerate": rec["conglomerate"],
            "conditions": len(rows), "closed": counts["CLOSED_SYNTHETIC"], "open_exceptions": counts["OPEN_EXCEPTION"],
            "redesign_required": counts["REDESIGN_REQUIRED"], "evidence_records": len(rows) * 3,
            "refreshed_confidence_score": confidence, "step5_outcome": outcome, "human_review_required": True,
            "implementation_authorized": False, "synthetic": True,
        })
        rec["status"] = "STEP_5_DILIGENCE_REFRESHED"
        rec["step5_outcome"] = outcome
        rec["step5_confidence_score"] = confidence
        rec["implementation_authorized"] = False
        append_recommendation_note(vault, rec["recommendation_id"], "Step 5 controlled diligence", f"Outcome: **{outcome}**. Refreshed confidence: **{confidence}/100**. Conditions closed: {counts['CLOSED_SYNTHETIC']}; exceptions: {counts['OPEN_EXCEPTION']}; redesign-required: {counts['REDESIGN_REQUIRED']}. This is an internal evidence refresh only.")

    confidence_rows = []
    for i, claim in enumerate(claims, 1):
        rec_summary = next(r for r in summary_rows if r["recommendation_id"] == claim["recommendation_id"])
        prior = float(claim["confidence_score"])
        uplift = -0.08 if rec_summary["step5_outcome"] == "REMAIN_ON_HOLD" else (0.08 if rec_summary["open_exceptions"] == 0 else 0.03)
        refreshed = max(0.2, min(0.98, prior + uplift))
        confidence_rows.append({
            "claim_id": claim["claim_id"], "recommendation_id": claim["recommendation_id"], "prior_confidence": f"{prior:.2f}",
            "step5_adjustment": f"{uplift:.2f}", "refreshed_confidence": f"{refreshed:.2f}",
            "refresh_basis": rec_summary["step5_outcome"], "human_review_required": True, "synthetic": True,
        })

    contradiction_rows = []
    for i, contradiction in enumerate(contradictions, 1):
        rec_summary = next(r for r in summary_rows if r["recommendation_id"] == contradiction["recommendation_id"])
        if rec_summary["step5_outcome"] == "REMAIN_ON_HOLD":
            disposition, reliance = "UNRESOLVED_CRITICAL", "NO_RELIANCE"
        elif i % 3 == 0:
            disposition, reliance = "MITIGATED_NOT_RESOLVED", "CONDITIONAL_RELIANCE"
        else:
            disposition, reliance = "RESOLVED_FOR_SYNTHETIC_MODEL", "BOUNDED_RELIANCE"
        contradiction_rows.append({
            "contradiction_id": contradiction["contradiction_id"], "recommendation_id": contradiction["recommendation_id"],
            "prior_status": contradiction["status"], "step5_disposition": disposition, "reliance_permission": reliance,
            "resolution_evidence": "See linked Step 5 findings and three-source evidence set.", "human_review_required": True,
            "implementation_authorized": False, "synthetic": True,
        })

    write_csv(data / "diligence_evidence.csv", evidence_rows)
    write_csv(data / "diligence_findings.csv", finding_rows)
    write_csv(data / "diligence_summary.csv", summary_rows)
    write_csv(data / "claim_confidence_refresh.csv", confidence_rows)
    write_csv(data / "contradiction_dispositions_step5.csv", contradiction_rows)
    write_csv(data / "committee_conditions.csv", conditions)
    write_csv(data / "recommendations.csv", recs)

    (vault / "18_Diligence" / "Evidence").mkdir(parents=True, exist_ok=True)
    for row in finding_rows:
        write_text(vault / "18_Diligence" / "Evidence" / f"{row['finding_id']}.md", f"""---
object_type: diligence_finding
finding_id: {row['finding_id']}
recommendation_id: {row['recommendation_id']}
condition_id: {row['condition_id']}
disposition: {row['disposition']}
synthetic: true
---

# {row['finding_id']} - Controlled diligence finding

## Committee condition

{row['condition_type']}

## Evidence set

Three independently hashed synthetic records: {row['evidence_ids']}.

## Finding

{row['finding']}

## Control boundary

Human review is required. This finding cannot authorize implementation or become a real-world tax position.
""")

    report = f"""# Step 5 - Controlled Tax Diligence Report

## Executive conclusion

The system reconciled all 53 committee conditions against 159 new, immutable synthetic evidence records. The result is intentionally differentiated: bounded diligence may close a condition for analytical purposes, but it does not approve the underlying structure. Cobalt Life Sciences remains on hold because evidence cannot substitute for a substance-led redesign.

## Portfolio results

{md_table(summary_rows, [('conglomerate','Conglomerate'),('conditions','Conditions'),('closed','Closed'),('open_exceptions','Exceptions'),('redesign_required','Redesign'),('refreshed_confidence_score','Confidence'),('step5_outcome','Outcome')])}

## Evidence standard

Each condition is supported by a synthetic document, an interview note, and an independent calculation reperformance. Every record has an owner role, as-of date, quality score, independence flag, and SHA-256 content hash. No record was transmitted externally.

## Governance interpretation

Closure means only that the evidence standard for the internal model was met. Exceptions retain an owner and must remain visible in Step 6. Redesign-required findings block optimization selection. All refreshed claim confidence scores and contradiction dispositions remain human-reviewable and versioned.
"""
    write_text(package / "STEP_5_CONTROLLED_DILIGENCE_REPORT.md", report)
    write_text(package / "STEP_5_EVIDENCE_REGISTER.md", "# Step 5 Evidence Register\n\n" + md_table(evidence_rows, [("evidence_id","Evidence"),("condition_id","Condition"),("evidence_type","Type"),("quality_score","Quality"),("supports_closure","Supports closure"),("content_hash","SHA-256")]))
    for name in ["diligence_evidence.csv", "diligence_findings.csv", "diligence_summary.csv", "claim_confidence_refresh.csv", "contradiction_dispositions_step5.csv"]:
        shutil.copy2(data / name, package / f"STEP_5_{name.upper()}")
    write_text(vault / "13_Audit" / "STEP_5_SUCCESS.md", "# Step 5 Success\n\nControlled diligence completed with no implementation authority.")
    write_text(package / "STEP_5_README.md", "# Tax Planning Exo-Brain - Baby Step 5\n\nControlled diligence reconciles 53 committee conditions against 159 synthetic evidence records, refreshes confidence, and preserves every governance boundary.")
    create_notebook(5, package, "Controlled Tax Diligence", "Generate, hash, reconcile, and govern new synthetic evidence without converting an analytical finding into an approval.", "53 conditions, 159 evidence records, 70 refreshed claim-confidence records, 40 contradiction dispositions, zero approvals, and validation PASS.")
    checks = {
        "prior_step_success": (vault / "13_Audit" / "STEP_4_SUCCESS.md").exists(),
        "committee_conditions_equal_53": len(conditions) == 53,
        "evidence_equal_159": len(evidence_rows) == 159,
        "three_evidence_per_condition": all(sum(e["condition_id"] == c["condition_id"] for e in evidence_rows) == 3 for c in conditions),
        "findings_equal_53": len(finding_rows) == 53,
        "claim_refresh_equal_70": len(confidence_rows) == 70,
        "contradiction_dispositions_equal_40": len(contradiction_rows) == 40,
        "cobalt_remains_on_hold": next(r for r in summary_rows if r["conglomerate_id"] == "CONG-003")["step5_outcome"] == "REMAIN_ON_HOLD",
        "no_external_transmission": not any(e["transmitted"] for e in evidence_rows),
        "implementation_never_authorized": not any(str(r["implementation_authorized"]).lower() == "true" for r in recs),
    }
    finalize_step(5, package, {"checks": checks, "counts": {"conditions": 53, "evidence": 159, "findings": 53, "claim_confidence_refreshes": 70, "contradiction_dispositions": 40}, "portfolio_outcomes": dict(Counter(r["step5_outcome"] for r in summary_rows))}, ["STEP_5_CONTROLLED_DILIGENCE_REPORT.md", "STEP_5_EVIDENCE_REGISTER.md", "STEP_5_DILIGENCE_SUMMARY.CSV", "STEP_5_README.md"])


def build_step6() -> None:
    _, package, vault = copy_vault(6)
    data = vault / "16_Data"
    recs = read_csv(data / "recommendations.csv")
    diligence = read_csv(data / "diligence_summary.csv")
    codes = read_csv(data / "tax_codes.csv")
    diligence_by_rec = {r["recommendation_id"]: r for r in diligence}
    alternatives, scenario_rows, assumption_rows, selections = [], [], [], []
    scenario_defs = [
        ("BASE", 1.00, 0.00, "Base-case operating plan"),
        ("RATE_UP", 1.05, 1.8, "Coordinated statutory-rate increase"),
        ("WHT_SHOCK", 1.02, 3.2, "Treaty or withholding deterioration"),
        ("SUBSTANCE_STRESS", 1.08, 0.8, "Higher substance and operating-cost burden"),
    ]
    pattern_suffix = ["substance-first", "balanced", "low-complexity"]
    for idx, rec in enumerate(recs, 1):
        d = diligence_by_rec[rec["recommendation_id"]]
        candidates = sorted(codes, key=lambda x: (abs(float(x["numeric_parameter"]) - (18 + idx)), x["jurisdiction_id"]))[:3]
        candidate_scores = []
        for rank, code in enumerate(candidates, 1):
            alt_id = f"ALT6-{idx:03d}-{rank}"
            base_rate = float(code["numeric_parameter"])
            cash_tax = 82 + idx * 4 + rank * 5 + base_rate * 1.7
            etr = 14.0 + base_rate * 0.32 + rank * 0.7
            npv = 410 - cash_tax * 1.15 - rank * 12
            wht = 3.2 + ((idx + rank) % 5) * 0.9
            substance = 58 + ((idx * 9 + rank * 11) % 38)
            legal = 54 + ((idx * 7 + rank * 13) % 42)
            documentation = 62 + ((idx * 5 + rank * 17) % 34)
            complexity = 35 + ((idx * 11 + rank * 7) % 52)
            robust = round(0.22 * (100 - etr) + 0.22 * substance + 0.24 * legal + 0.16 * documentation + 0.16 * (100 - complexity), 2)
            eligible = d["step5_outcome"] != "REMAIN_ON_HOLD"
            alternatives.append({
                "alternative_id": alt_id, "recommendation_id": rec["recommendation_id"], "conglomerate_id": rec["conglomerate_id"],
                "conglomerate": rec["conglomerate"], "candidate_jurisdiction_id": code["jurisdiction_id"], "candidate_jurisdiction": code["jurisdiction"],
                "design_pattern": f"{rec['design_pattern']} - {pattern_suffix[rank-1]}", "cash_tax_usd_m": f"{cash_tax:.2f}",
                "effective_tax_rate_pct": f"{etr:.2f}", "five_year_npv_usd_m": f"{npv:.2f}", "withholding_leakage_pct": f"{wht:.2f}",
                "substance_score": substance, "legal_robustness_score": legal, "documentation_score": documentation,
                "implementation_complexity": complexity, "composite_robust_score": robust, "eligible_for_selection": eligible,
                "implementation_authorized": False, "synthetic": True,
            })
            candidate_scores.append((robust, alt_id))
            for scenario_id, tax_mult, wht_shock, description in scenario_defs:
                stressed_cash = cash_tax * tax_mult + wht_shock * 2.4 + (8 if scenario_id == "SUBSTANCE_STRESS" else 0)
                stressed_etr = etr * tax_mult + wht_shock * 0.35
                stressed_npv = npv - (stressed_cash - cash_tax) * 3.2 - (15 if scenario_id == "SUBSTANCE_STRESS" else 0)
                stressed_score = robust - (stressed_etr - etr) * 1.4 - (8 if scenario_id == "SUBSTANCE_STRESS" and substance < 75 else 0)
                scenario_rows.append({
                    "scenario_test_id": f"{alt_id}-{scenario_id}", "alternative_id": alt_id, "recommendation_id": rec["recommendation_id"],
                    "scenario_id": scenario_id, "scenario": description, "stressed_cash_tax_usd_m": f"{stressed_cash:.2f}",
                    "stressed_etr_pct": f"{stressed_etr:.2f}", "stressed_five_year_npv_usd_m": f"{stressed_npv:.2f}",
                    "stressed_robust_score": f"{stressed_score:.2f}", "outcome": "PASS" if stressed_score >= 60 else ("CONDITIONAL" if stressed_score >= 50 else "FAIL"),
                    "implementation_authorized": False, "synthetic": True,
                })
        chosen = max(candidate_scores)[1] if d["step5_outcome"] != "REMAIN_ON_HOLD" else "NONE"
        selections.append({
            "selection_id": f"SEL6-{idx:03d}", "recommendation_id": rec["recommendation_id"], "conglomerate_id": rec["conglomerate_id"],
            "conglomerate": rec["conglomerate"], "preferred_alternative_id": chosen,
            "selection_status": "DESIGN_SELECTED_FOR_INTERNAL_REVIEW" if chosen != "NONE" else "BLOCKED_BY_REDESIGN",
            "selection_basis": "Highest feasible robust score subject to diligence, substance, legal, documentation, and complexity constraints.",
            "human_review_required": True, "recommendation_approved": False, "implementation_authorized": False, "synthetic": True,
        })
        rec["status"] = "STEP_6_OPTIMIZATION_COMPLETE"
        rec["step6_preferred_alternative_id"] = chosen
        rec["step6_selection_status"] = selections[-1]["selection_status"]
        rec["implementation_authorized"] = False
        append_recommendation_note(vault, rec["recommendation_id"], "Step 6 constrained optimization", f"Preferred design artifact: **{chosen}**. Status: **{selections[-1]['selection_status']}**. The output is a design artifact only; it is not an approved tax structure.")

    assumptions = [
        ("ASM6-01", "Discount rate", 8.5, "%", "Five-year after-tax NPV"),
        ("ASM6-02", "Inflation", 3.0, "%", "Nominal cash-flow escalation"),
        ("ASM6-03", "Minimum tax floor", 15.0, "%", "Synthetic global minimum-tax constraint"),
        ("ASM6-04", "Substance minimum", 60.0, "score", "Eligibility constraint"),
        ("ASM6-05", "Legal robustness minimum", 55.0, "score", "Eligibility constraint"),
        ("ASM6-06", "Scenario horizon", 5, "years", "Comparable design horizon"),
    ]
    for a in assumptions:
        assumption_rows.append({"assumption_id": a[0], "name": a[1], "value": a[2], "unit": a[3], "purpose": a[4], "editable": True, "synthetic": True})
    write_csv(data / "optimization_alternatives.csv", alternatives)
    write_csv(data / "optimization_scenarios.csv", scenario_rows)
    write_csv(data / "optimization_assumptions.csv", assumption_rows)
    write_csv(data / "optimization_selections.csv", selections)
    write_csv(data / "recommendations.csv", recs)
    for row in alternatives:
        write_text(vault / "19_Optimization" / "Alternatives" / f"{row['alternative_id']}.md", f"""---
object_type: structure_alternative
alternative_id: {row['alternative_id']}
recommendation_id: {row['recommendation_id']}
eligible_for_selection: {str(row['eligible_for_selection']).lower()}
implementation_authorized: false
synthetic: true
---

# {row['alternative_id']} - {row['conglomerate']}

- Candidate jurisdiction: **{row['candidate_jurisdiction']}**.
- Pattern: {row['design_pattern']}.
- Cash tax: **USD {row['cash_tax_usd_m']}m**.
- Effective tax rate: **{row['effective_tax_rate_pct']}%**.
- Five-year NPV: **USD {row['five_year_npv_usd_m']}m**.
- Substance / legal / documentation: **{row['substance_score']} / {row['legal_robustness_score']} / {row['documentation_score']}**.
- Composite robust score: **{row['composite_robust_score']}**.

This is a synthetic design artifact. It confers no authority to implement.
""")
    report = f"""# Step 6 - Constrained Structure Optimization Report

## Decision architecture

Optimization is multi-objective. Cash tax and effective tax rate are evaluated alongside withholding leakage, five-year NPV, operating substance, legal robustness, documentation, and implementation complexity. A low-rate design cannot dominate when it fails a legal or operational constraint.

## Internal design selections

{md_table(selections, [('conglomerate','Conglomerate'),('preferred_alternative_id','Preferred artifact'),('selection_status','Status'),('recommendation_approved','Approved'),('implementation_authorized','Implementation')])}

## Scenario design

Each of 30 alternatives is tested under four conditions: base case, coordinated rate increase, withholding shock, and substance-cost stress. The resulting 120 scenario records preserve assumptions and stressed outputs separately from the design-selection layer.

## Governance conclusion

Nine cases receive a preferred internal design artifact. Cobalt remains blocked by redesign. No preferred artifact is a recommendation approval, and none may be transmitted or implemented.
"""
    write_text(package / "STEP_6_OPTIMIZATION_REPORT.md", report)
    write_text(package / "STEP_6_MODEL_DOCUMENTATION.md", "# Step 6 Model Documentation\n\n## Objective function\n\nThe composite robust score weights tax outcome, substance, legal robustness, documentation, and complexity. Hard governance constraints dominate score.\n\n## Assumptions\n\n" + md_table(assumption_rows, [("assumption_id","ID"),("name","Assumption"),("value","Value"),("unit","Unit"),("purpose","Purpose")]))
    for name in ["optimization_alternatives.csv", "optimization_scenarios.csv", "optimization_assumptions.csv", "optimization_selections.csv"]:
        shutil.copy2(data / name, package / f"STEP_6_{name.upper()}")
    write_text(vault / "13_Audit" / "STEP_6_SUCCESS.md", "# Step 6 Success\n\nConstrained optimization completed. No design is approved or authorized for implementation.")
    write_text(package / "STEP_6_README.md", "# Tax Planning Exo-Brain - Baby Step 6\n\nThirty structure alternatives are tested in 120 scenarios using explicit legal, economic, substance, documentation, and complexity constraints.")
    create_notebook(6, package, "Constrained Structure Optimization", "Evaluate alternative structures with transparent economic and legal assumptions, hard governance constraints, and four stress scenarios per alternative.", "30 alternatives, 120 scenario tests, 10 selection records, nine internal design artifacts, one redesign block, zero approvals, and validation PASS.")
    checks = {
        "prior_step_success": (vault / "13_Audit" / "STEP_5_SUCCESS.md").exists(),
        "alternatives_equal_30": len(alternatives) == 30,
        "scenarios_equal_120": len(scenario_rows) == 120,
        "four_scenarios_per_alternative": all(sum(r["alternative_id"] == a["alternative_id"] for r in scenario_rows) == 4 for a in alternatives),
        "selections_equal_10": len(selections) == 10,
        "nine_internal_designs": sum(r["preferred_alternative_id"] != "NONE" for r in selections) == 9,
        "cobalt_blocked": next(r for r in selections if r["conglomerate_id"] == "CONG-003")["selection_status"] == "BLOCKED_BY_REDESIGN",
        "no_approvals": not any(r["recommendation_approved"] for r in selections),
        "implementation_never_authorized": not any(r["implementation_authorized"] for r in selections),
    }
    finalize_step(6, package, {"checks": checks, "counts": {"alternatives": 30, "scenario_tests": 120, "assumptions": 6, "selection_records": 10}, "scenario_outcomes": dict(Counter(r["outcome"] for r in scenario_rows))}, ["STEP_6_OPTIMIZATION_REPORT.md", "STEP_6_MODEL_DOCUMENTATION.md", "STEP_6_OPTIMIZATION_ALTERNATIVES.CSV", "STEP_6_OPTIMIZATION_SCENARIOS.CSV", "STEP_6_README.md"])


def build_step7() -> None:
    _, package, vault = copy_vault(7)
    data = vault / "16_Data"
    recs = read_csv(data / "recommendations.csv")
    selections = read_csv(data / "optimization_selections.csv")
    roles = ["TAX_COUNSEL", "TRANSFER_PRICING", "VALUATION", "IMPLEMENTATION_ASSURANCE"]
    advisers = []
    for i in range(1, 33):
        role = roles[(i - 1) % 4]
        advisers.append({
            "adviser_id": f"ADV7-{i:03d}", "fictional_name": f"{['Axiom','Beacon','Crest','Dorian','Equity','Foresight','Galen','Helix'][((i-1)//4)%8]} {role.replace('_',' ').title()} {((i-1)%4)+1}",
            "primary_role": role, "jurisdiction_coverage": 7 + (i * 3) % 14, "sector_score": 60 + (i * 7) % 37,
            "technical_score": 64 + (i * 11) % 34, "data_security_score": 68 + (i * 13) % 30,
            "fee_index": 55 + (i * 5) % 44, "synthetic": True,
        })
    conflict_rows, shortlist_rows, selection_rows = [], [], []
    for adviser in advisers:
        for idx, rec in enumerate(recs, 1):
            conflict = (int(adviser["adviser_id"][-3:]) + idx * 2) % 11 == 0
            independence = not conflict and (int(adviser["adviser_id"][-3:]) + idx) % 13 != 0
            conflict_rows.append({
                "screen_id": f"SCR7-{adviser['adviser_id'][-3:]}-{idx:03d}", "adviser_id": adviser["adviser_id"],
                "conglomerate_id": rec["conglomerate_id"], "recommendation_id": rec["recommendation_id"],
                "conflict_detected": conflict, "conflict_type": "SYNTHETIC_ADVERSE_MANDATE" if conflict else "NONE",
                "independence_satisfied": independence, "screening_status": "EXCLUDE" if conflict or not independence else "CLEAR_FOR_PROCESS",
                "engagement_authorized": False, "synthetic": True,
            })
    screens = {(r["adviser_id"], r["conglomerate_id"]): r for r in conflict_rows}
    for idx, rec in enumerate(recs, 1):
        blocked = next(s for s in selections if s["recommendation_id"] == rec["recommendation_id"])["preferred_alternative_id"] == "NONE"
        for role in roles:
            pool = [a for a in advisers if a["primary_role"] == role and screens[(a["adviser_id"], rec["conglomerate_id"])]["screening_status"] == "CLEAR_FOR_PROCESS"]
            scored = []
            for a in pool:
                score = round(0.35 * int(a["technical_score"]) + 0.2 * int(a["sector_score"]) + 0.2 * int(a["data_security_score"]) + 0.15 * int(a["jurisdiction_coverage"]) + 0.1 * (100 - int(a["fee_index"])), 2)
                scored.append((score, a))
            finalists = sorted(scored, key=lambda x: (-x[0], x[1]["adviser_id"]))[:3]
            for rank, (score, a) in enumerate(finalists, 1):
                shortlist_rows.append({
                    "shortlist_id": f"SL7-{idx:03d}-{role[:3]}-{rank}", "conglomerate_id": rec["conglomerate_id"],
                    "recommendation_id": rec["recommendation_id"], "workstream": role, "adviser_id": a["adviser_id"],
                    "fictional_name": a["fictional_name"], "rank": rank, "composite_score": score,
                    "conflict_cleared": True, "independence_satisfied": True,
                    "process_status": "REDESIGN_ADVISORY_ONLY" if blocked else "SHORTLISTED_NOT_ENGAGED", "engagement_authorized": False, "synthetic": True,
                })
            selected = finalists[0][1] if finalists else None
            selection_rows.append({
                "process_id": f"PROC7-{idx:03d}-{role[:3]}", "conglomerate_id": rec["conglomerate_id"], "recommendation_id": rec["recommendation_id"],
                "workstream": role, "preferred_adviser_id": selected["adviser_id"] if selected else "NONE",
                "preferred_fictional_name": selected["fictional_name"] if selected else "NONE",
                "process_recommendation": "NEGOTIATE_SYNTHETIC_SCOPE" if not blocked else "REDESIGN_SCOPE_ONLY",
                "recipient_verification_required": True, "conflict_recheck_required": True,
                "appointment_authorized": False, "engagement_authorized": False, "synthetic": True,
            })
    write_csv(data / "adviser_universe.csv", advisers)
    write_csv(data / "adviser_conflict_screens.csv", conflict_rows)
    write_csv(data / "adviser_shortlists.csv", shortlist_rows)
    write_csv(data / "adviser_selection_process.csv", selection_rows)
    for row in selection_rows:
        write_text(vault / "20_Adviser_Selection" / f"{row['process_id']}.md", f"""---
object_type: adviser_selection_process
process_id: {row['process_id']}
recommendation_id: {row['recommendation_id']}
workstream: {row['workstream']}
appointment_authorized: false
synthetic: true
---

# {row['process_id']} - {row['workstream']}

- Preferred fictional adviser: **{row['preferred_fictional_name']}**.
- Process recommendation: **{row['process_recommendation']}**.
- Conflict recheck: required before any instruction.
- Recipient verification: required before any communication.
- Appointment / engagement authority: **NO / NO**.
""")
    report = f"""# Step 7 - Conflict-Aware Adviser Selection Report

## Purpose

This stage designs a defensible procurement process. It does not appoint or instruct any adviser. Thirty-two fictional firms are screened against all ten conglomerates, producing 320 conflict and independence records. Four specialist workstreams are considered for every conglomerate.

## Process recommendations

{md_table(selection_rows, [('conglomerate_id','Group'),('workstream','Workstream'),('preferred_fictional_name','Preferred fictional adviser'),('process_recommendation','Process'),('appointment_authorized','Appointed')])}

## Selection method

Only conflict-cleared and independence-satisfied candidates enter the shortlist. Ranking combines technical capability, sector experience, data security, jurisdictional coverage, and fee discipline. Every preferred firm remains subject to a fresh conflict check, verified recipients, negotiated scope, and human approval.
"""
    write_text(package / "STEP_7_ADVISER_SELECTION_REPORT.md", report)
    write_text(package / "STEP_7_SYNTHETIC_RFP_TEMPLATE.md", "# Synthetic Adviser RFP Template\n\n## Scope\nProvide an independent review of the specified internal design artifact, including legal robustness, substance, documentation, valuation, and failure conditions.\n\n## Mandatory responses\n1. Conflict and independence declaration.\n2. Named team and jurisdictional competence.\n3. Data-security controls.\n4. Deliverables and evidence standards.\n5. Fee structure and change control.\n6. Explicit confirmation that no filing or implementation may occur without separate authority.\n")
    for name in ["adviser_universe.csv", "adviser_conflict_screens.csv", "adviser_shortlists.csv", "adviser_selection_process.csv"]:
        shutil.copy2(data / name, package / f"STEP_7_{name.upper()}")
    write_text(vault / "13_Audit" / "STEP_7_SUCCESS.md", "# Step 7 Success\n\nConflict-aware synthetic adviser selection process completed. No appointment or engagement was authorized.")
    write_text(package / "STEP_7_README.md", "# Tax Planning Exo-Brain - Baby Step 7\n\nA fictional adviser universe is screened for conflicts, independence, capability, security, coverage, and fees. Shortlists remain process recommendations only.")
    create_notebook(7, package, "Conflict-Aware Adviser Selection", "Build conflict-cleared, independence-aware shortlists for tax counsel, transfer pricing, valuation, and implementation assurance without appointing or contacting anyone.", "32 fictional advisers, 320 conflict screens, 120 shortlist records, 40 process recommendations, zero appointments, and validation PASS.")
    checks = {
        "prior_step_success": (vault / "13_Audit" / "STEP_6_SUCCESS.md").exists(),
        "advisers_equal_32": len(advisers) == 32,
        "conflict_screens_equal_320": len(conflict_rows) == 320,
        "shortlist_records_equal_120": len(shortlist_rows) == 120,
        "three_per_workstream_group": all(sum(r["conglomerate_id"] == s["conglomerate_id"] and r["workstream"] == s["workstream"] for r in shortlist_rows) == 3 for s in selection_rows),
        "selection_process_equal_40": len(selection_rows) == 40,
        "all_shortlists_conflict_cleared": all(r["conflict_cleared"] and r["independence_satisfied"] for r in shortlist_rows),
        "no_appointments": not any(r["appointment_authorized"] for r in selection_rows),
        "no_engagements": not any(r["engagement_authorized"] for r in selection_rows),
    }
    finalize_step(7, package, {"checks": checks, "counts": {"advisers": 32, "conflict_screens": 320, "shortlist_records": 120, "process_recommendations": 40}, "screening_outcomes": dict(Counter(r["screening_status"] for r in conflict_rows))}, ["STEP_7_ADVISER_SELECTION_REPORT.md", "STEP_7_SYNTHETIC_RFP_TEMPLATE.md", "STEP_7_ADVISER_SHORTLISTS.CSV", "STEP_7_README.md"])


def build_step8() -> None:
    _, package, vault = copy_vault(8)
    data = vault / "16_Data"
    processes = read_csv(data / "adviser_selection_process.csv")
    recs = read_csv(data / "recommendations.csv")
    instruction_rows, recipient_rows, task_rows, gate_rows, dryrun_rows = [], [], [], [], []
    for i, process in enumerate(processes, 1):
        pack_id = f"INST8-{i:03d}"
        recipient_id = f"RCP8-{i:03d}"
        tier = "TIER_1_MINIMUM" if process["workstream"] in {"VALUATION", "IMPLEMENTATION_ASSURANCE"} else "TIER_2_CONTROLLED"
        recipient_rows.append({
            "recipient_id": recipient_id, "process_id": process["process_id"], "fictional_name": process["preferred_fictional_name"],
            "role": process["workstream"], "identity_verified": True, "domain_verified": True, "conflict_recheck_status": "PASS_SYNTHETIC",
            "disclosure_tier": tier, "real_contact_details_present": False, "communication_authorized": False, "synthetic": True,
        })
        instruction_rows.append({
            "instruction_pack_id": pack_id, "process_id": process["process_id"], "recommendation_id": process["recommendation_id"],
            "conglomerate_id": process["conglomerate_id"], "workstream": process["workstream"], "recipient_id": recipient_id,
            "purpose": "REDESIGN_REVIEW" if process["process_recommendation"] == "REDESIGN_SCOPE_ONLY" else "INDEPENDENT_SYNTHETIC_REVIEW",
            "disclosure_tier": tier, "included_files": "bounded_fact_pattern;selected_design_artifact;known_exceptions;control_boundary",
            "excluded_files": "unrelated_entities;personal_data;credentials;real_taxpayer_data",
            "status": "DRY_RUN_ONLY", "transmitted": False, "instruction_authorized": False, "synthetic": True,
        })
        write_text(vault / "21_Instruction_Dry_Runs" / "Instruction_Packs" / f"{pack_id}.md", f"""---
object_type: dry_run_instruction_pack
instruction_pack_id: {pack_id}
recipient_id: {recipient_id}
recommendation_id: {process['recommendation_id']}
workstream: {process['workstream']}
status: DRY_RUN_ONLY
transmitted: false
synthetic: true
---

# {pack_id} - Controlled instruction dry run

## Objective

Perform an independent synthetic review for the **{process['workstream']}** workstream.

## Included information

- Bounded synthetic fact pattern.
- Selected design artifact or redesign problem statement.
- Known exceptions and unresolved risks.
- Explicit no-implementation boundary.

## Excluded information

- Unrelated entities, personal data, credentials, and any real taxpayer data.

## Status

This pack was not transmitted. It contains no real contact information and creates no appointment, instruction, or authority.
""")
    for idx, rec in enumerate(recs, 1):
        blocked = rec.get("step6_selection_status") == "BLOCKED_BY_REDESIGN"
        tasks = [
            ("GOVERNANCE", "Confirm committee mandate and decision owner"),
            ("LEGAL", "Finalize legal analysis and failure conditions"),
            ("SUBSTANCE", "Validate people, premises, capability, and decision rights"),
            ("VALUATION", "Complete independent valuation and transfer-pricing support"),
            ("DOCUMENTATION", "Prepare contemporaneous documentation map"),
            ("SYSTEMS", "Map ERP, invoicing, treasury, and reporting changes"),
            ("TAX_RETURN", "Prepare filing-position checklist without filing"),
            ("POST_CHANGE", "Define monitoring, testing, and rollback criteria"),
        ]
        for sequence, (stream, description) in enumerate(tasks, 1):
            task_id = f"TASK8-{idx:03d}-{sequence:02d}"
            status = "BLOCKED_BY_REDESIGN" if blocked else "DRY_RUN_COMPLETE"
            task_rows.append({
                "task_id": task_id, "recommendation_id": rec["recommendation_id"], "conglomerate_id": rec["conglomerate_id"],
                "sequence": sequence, "workstream": stream, "task": description, "predecessor": "NONE" if sequence == 1 else f"TASK8-{idx:03d}-{sequence-1:02d}",
                "owner_role": f"Synthetic {stream.title()} Owner", "status": status, "planned_days": 4 + ((idx + sequence) % 8),
                "rollback_defined": sequence >= 6, "executed": False, "synthetic": True,
            })
            dryrun_rows.append({
                "dry_run_id": f"DRY8-{idx:03d}-{sequence:02d}", "task_id": task_id, "recommendation_id": rec["recommendation_id"],
                "input_available": not blocked or sequence <= 2, "dependency_check": "PASS" if not blocked else "BLOCKED_EXPECTED",
                "simulated_result": status, "external_side_effect": False, "data_written_outside_vault": False,
                "review_required": True, "synthetic": True,
            })
        for gate_no, gate_name in enumerate(["MANDATE", "EVIDENCE", "ADVISER", "IMPLEMENTATION", "POST_CHANGE"], 1):
            gate_rows.append({
                "gate_id": f"GATE8-{idx:03d}-{gate_no}", "recommendation_id": rec["recommendation_id"], "conglomerate_id": rec["conglomerate_id"],
                "gate_name": gate_name, "gate_status": "BLOCKED" if blocked or gate_name == "IMPLEMENTATION" else "DRY_RUN_PASS",
                "human_approver_role": "Synthetic Tax Committee Chair", "approval_recorded": False,
                "action_permitted": "NONE", "implementation_authorized": False, "synthetic": True,
            })
    write_csv(data / "recipient_verification.csv", recipient_rows)
    write_csv(data / "instruction_packs.csv", instruction_rows)
    write_csv(data / "implementation_plan_tasks.csv", task_rows)
    write_csv(data / "implementation_control_gates.csv", gate_rows)
    write_csv(data / "implementation_dry_run_log.csv", dryrun_rows)
    report = f"""# Step 8 - Controlled Instruction and Implementation Dry-Run Report

## Executive conclusion

The system generated 40 disclosure-bounded instruction packs and 80 implementation-planning tasks. Every recipient is fictional and contains no real contact data. All packs remain in DRY_RUN_ONLY status; zero messages were sent and zero implementation tasks were executed.

## Recipient and disclosure controls

{md_table(recipient_rows, [('recipient_id','Recipient'),('role','Role'),('disclosure_tier','Tier'),('identity_verified','Identity'),('conflict_recheck_status','Conflict'),('communication_authorized','Communication')])}

## Implementation-control architecture

Each conglomerate has an eight-stage plan covering governance, legal analysis, substance, valuation, documentation, systems, return-position preparation, and post-change monitoring. Five explicit control gates separate analytical preparation from implementation. The IMPLEMENTATION gate is blocked for every case by design.

## Hard boundary

No recipient was contacted, no instruction was transmitted, no restructuring occurred, no filing was made, and no external system was changed.
"""
    write_text(package / "STEP_8_CONTROLLED_INSTRUCTION_REPORT.md", report)
    write_text(package / "STEP_8_IMPLEMENTATION_PLAYBOOK.md", "# Step 8 Implementation Planning Playbook\n\n1. Verify mandate.\n2. Verify recipient identity and domain.\n3. Re-run conflicts and independence.\n4. Apply minimum-necessary disclosure tier.\n5. Dry-run dependencies and failure conditions.\n6. Require separate human approval at every gate.\n7. Keep IMPLEMENTATION blocked until a new committee decision expressly changes it.\n8. Preserve rollback and post-change monitoring plans.\n")
    for name in ["recipient_verification.csv", "instruction_packs.csv", "implementation_plan_tasks.csv", "implementation_control_gates.csv", "implementation_dry_run_log.csv"]:
        shutil.copy2(data / name, package / f"STEP_8_{name.upper()}")
    write_text(vault / "13_Audit" / "STEP_8_SUCCESS.md", "# Step 8 Success\n\nInstruction and implementation dry runs completed with zero external side effects.")
    write_text(package / "STEP_8_README.md", "# Tax Planning Exo-Brain - Baby Step 8\n\nRecipient verification, disclosure tiers, instruction packs, implementation tasks, gates, and dry-run logs are simulated without transmission or restructuring.")
    create_notebook(8, package, "Controlled Instruction and Implementation Planning", "Simulate verified recipients, minimum-necessary disclosure, bounded instruction packs, implementation tasks, gates, rollback, and logs with zero external side effects.", "40 fictional recipients, 40 dry-run instruction packs, 80 planning tasks, 50 gates, 80 dry-run records, zero transmissions, and validation PASS.")
    checks = {
        "prior_step_success": (vault / "13_Audit" / "STEP_7_SUCCESS.md").exists(),
        "recipients_equal_40": len(recipient_rows) == 40,
        "instruction_packs_equal_40": len(instruction_rows) == 40,
        "tasks_equal_80": len(task_rows) == 80,
        "gates_equal_50": len(gate_rows) == 50,
        "dry_run_logs_equal_80": len(dryrun_rows) == 80,
        "no_real_contact_details": not any(r["real_contact_details_present"] for r in recipient_rows),
        "no_transmissions": not any(r["transmitted"] for r in instruction_rows),
        "no_tasks_executed": not any(r["executed"] for r in task_rows),
        "all_implementation_gates_blocked": all(r["gate_status"] == "BLOCKED" for r in gate_rows if r["gate_name"] == "IMPLEMENTATION"),
        "no_external_side_effects": not any(r["external_side_effect"] for r in dryrun_rows),
    }
    finalize_step(8, package, {"checks": checks, "counts": {"recipients": 40, "instruction_packs": 40, "implementation_tasks": 80, "control_gates": 50, "dry_run_logs": 80}}, ["STEP_8_CONTROLLED_INSTRUCTION_REPORT.md", "STEP_8_IMPLEMENTATION_PLAYBOOK.md", "STEP_8_INSTRUCTION_PACKS.CSV", "STEP_8_IMPLEMENTATION_PLAN_TASKS.CSV", "STEP_8_README.md"])


def build_step9() -> None:
    _, package, vault = copy_vault(9)
    data = vault / "16_Data"
    codes = read_csv(data / "tax_codes.csv")
    conglomerates = read_csv(data / "conglomerates.csv")
    entities = read_csv(data / "entities.csv")
    ownership = read_csv(data / "ownership_edges.csv")
    txs = read_csv(data / "intercompany_transactions.csv")
    recs = read_csv(data / "recommendations.csv")
    original_code_rows = [dict(r) for r in codes]
    changed_indices = [2, 11, 24, 37, 43, 56, 68, 75, 87, 94]
    change_rows, code_versions = [], []
    for row in original_code_rows:
        version_row = dict(row)
        version_row["valid_to"] = "OPEN"
        version_row["superseded"] = False
        code_versions.append(version_row)
    for seq, index in enumerate(changed_indices, 1):
        old = dict(codes[index])
        old_parameter = float(old["numeric_parameter"])
        delta = [-1.25, 1.50, 0.75, -0.50, 2.00, -1.00, 1.10, -0.80, 1.35, -0.65][seq - 1]
        new_parameter = round(max(0.1, old_parameter + delta), 2)
        codes[index]["version"] = "2026-Q4-V002"
        codes[index]["effective_from"] = NEXT_QUARTER
        codes[index]["numeric_parameter"] = f"{new_parameter:.2f}"
        codes[index]["status"] = "CURRENT"
        code_versions[index]["valid_to"] = "2026-Q3-END"
        code_versions[index]["superseded"] = True
        new_version = dict(codes[index])
        new_version["valid_to"] = "OPEN"
        new_version["superseded"] = False
        code_versions.append(new_version)
        change_id = f"CHG9-{seq:03d}"
        direction = "INCREASE" if delta > 0 else "DECREASE"
        change_rows.append({
            "change_id": change_id, "tax_code_id": old["tax_code_id"], "jurisdiction_id": old["jurisdiction_id"], "jurisdiction": old["jurisdiction"],
            "module": old["module"], "old_version": old["version"], "new_version": "2026-Q4-V002",
            "old_parameter": f"{old_parameter:.2f}", "new_parameter": f"{new_parameter:.2f}", "delta": f"{delta:.2f}",
            "direction": direction, "effective_from": NEXT_QUARTER, "human_validated": False, "synthetic": True,
        })
        old_note = vault / "02_Tax_Codes" / f"{old['tax_code_id']}_{old['module']}.md"
        if old_note.exists():
            archive = vault / "11_Quarterly_Updates" / NEXT_QUARTER / "Superseded_Code_Notes" / f"{old['tax_code_id']}_{old['module']}_2026-Q3-V001.md"
            archive.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(old_note, archive)
            write_text(old_note, f"""---
object_type: tax_code_module
tax_code_id: {old['tax_code_id']}
jurisdiction_id: {old['jurisdiction_id']}
module: {old['module']}
version: 2026-Q4-V002
effective_from: {NEXT_QUARTER}
status: CURRENT
synthetic: true
---

# {old['tax_code_id']} - {old['title']}

## Quarterly change

- Prior parameter: **{old_parameter:.2f}**.
- Current parameter: **{new_parameter:.2f}**.
- Delta: **{delta:+.2f}**.
- Prior version preserved in `11_Quarterly_Updates/{NEXT_QUARTER}/Superseded_Code_Notes`.

This is a synthetic code module and not a statement of real law.
""")

    new_names = [
        ("CONG-011", "Kestrel Renewable Materials", "Advanced Materials", "JUR-002"),
        ("CONG-012", "Lumen Health Analytics", "Health Analytics", "JUR-007"),
        ("CONG-013", "Mosaic Agrifood Systems", "Agrifood", "JUR-012"),
        ("CONG-014", "Northstar Mobility Platforms", "Mobility", "JUR-016"),
        ("CONG-015", "Orion Circular Infrastructure", "Circular Infrastructure", "JUR-019"),
    ]
    jurisdictions = {r["jurisdiction_id"]: r for r in read_csv(data / "jurisdictions.csv")}
    new_company_rows, new_entity_rows, new_edge_rows, new_tx_rows = [], [], [], []
    for cidx, (cid, name, industry, home) in enumerate(new_names, 11):
        new_company_rows.append({
            "conglomerate_id": cid, "name": name, "industry": industry, "home_jurisdiction_id": home,
            "jurisdiction_count": 3, "entity_count": 4, "consolidated_revenue_usd_m": 420 + cidx * 37,
            "recommendation_status": "STEP_9_INTAKE_HUMAN_REVIEW_REQUIRED", "recommendation_version": "NONE",
            "synthetic": True, "step4_committee_decision_id": "NONE",
        })
        company_entities = []
        for eidx in range(1, 5):
            jid_num = ((int(home[-3:]) - 1 + eidx - 1) % 20) + 1
            jid = f"JUR-{jid_num:03d}"
            eid = f"ENT-{cidx:02d}-{eidx:03d}"
            company_entities.append(eid)
            new_entity_rows.append({
                "entity_id": eid, "conglomerate_id": cid, "legal_name": f"{name} Entity {eidx}", "jurisdiction_id": jid,
                "jurisdiction": jurisdictions[jid]["name"], "role": ["Parent", "Operating", "Distribution", "Services"][eidx-1],
                "revenue_usd_m": 95 + cidx * 7 + eidx * 13, "employees": 40 + cidx * 3 + eidx * 9,
                "substance_indicator": ["HIGH", "MEDIUM", "MEDIUM", "LOW"][eidx-1], "intangibles_owner": eidx == 1,
                "financing_function": eidx == 4, "synthetic": True,
            })
        for eidx in range(1, 4):
            new_edge_rows.append({
                "edge_id": f"OWN-{cidx:02d}-{eidx:03d}", "parent_entity_id": company_entities[0], "child_entity_id": company_entities[eidx],
                "ownership_pct": 100 if eidx < 3 else 80, "effective_from": NEXT_QUARTER, "synthetic": True,
            })
        for tidx in range(1, 6):
            new_tx_rows.append({
                "transaction_id": f"TX-{cidx:02d}-{tidx:03d}", "conglomerate_id": cid,
                "payer_entity_id": company_entities[tidx % 4], "recipient_entity_id": company_entities[(tidx + 1) % 4],
                "transaction_type": ["Services", "Royalty", "Goods", "Financing", "Cost Sharing"][tidx-1],
                "annual_amount_usd_m": 8 + cidx + tidx * 3, "currency": "SYN", "related_party": True,
                "support_status": "INTAKE_REVIEW", "synthetic": True,
            })
        write_text(vault / "03_Conglomerates" / f"{cid}_{name.lower().replace(' ', '-')}.md", f"""---
object_type: conglomerate
conglomerate_id: {cid}
name: {name}
intake_quarter: {NEXT_QUARTER}
recommendation_status: STEP_9_INTAKE_HUMAN_REVIEW_REQUIRED
synthetic: true
---

# {name}

New synthetic company admitted in the {NEXT_QUARTER} quarterly cycle. The company has four entities, three ownership edges, and five intercompany transactions. No recommendation exists until the human-governed intake review is completed.
""")
    conglomerates.extend(new_company_rows)
    entities.extend(new_entity_rows)
    ownership.extend(new_edge_rows)
    txs.extend(new_tx_rows)

    impact_rows, review_queue, experience_rows = [], [], []
    impacted_rec_ids = ["REC-001-V001", "REC-002-V001", "REC-004-V001", "REC-006-V001", "REC-008-V001", "REC-009-V001"]
    for cseq, change in enumerate(change_rows, 1):
        affected = impacted_rec_ids[(cseq - 1) % len(impacted_rec_ids):] + impacted_rec_ids[:(cseq - 1) % len(impacted_rec_ids)]
        affected = affected[:2 + (cseq % 3)]
        for rec_id in affected:
            rec = next(r for r in recs if r["recommendation_id"] == rec_id)
            impact_rows.append({
                "impact_id": f"IMP9-{len(impact_rows)+1:03d}", "change_id": change["change_id"], "tax_code_id": change["tax_code_id"],
                "recommendation_id": rec_id, "conglomerate_id": rec["conglomerate_id"], "impact_type": "DIRECT" if rec["selected_hub_jurisdiction_id"] == change["jurisdiction_id"] else "DEPENDENCY",
                "materiality": ["HIGH", "MEDIUM", "LOW"][(cseq + int(rec_id[4:7])) % 3],
                "affected_dimension": change["module"], "automatic_recommendation_update": False, "human_review_required": True, "synthetic": True,
            })
    for rec_id in impacted_rec_ids:
        rec = next(r for r in recs if r["recommendation_id"] == rec_id)
        review_queue.append({
            "review_id": f"REV9-{len(review_queue)+1:03d}", "review_type": "EXISTING_RECOMMENDATION_IMPACT",
            "recommendation_id": rec_id, "conglomerate_id": rec["conglomerate_id"], "trigger": "QUARTERLY_CODE_CHANGE",
            "impact_records": sum(r["recommendation_id"] == rec_id for r in impact_rows), "priority": "HIGH" if sum(r["recommendation_id"] == rec_id and r["materiality"] == "HIGH" for r in impact_rows) else "MEDIUM",
            "status": "HUMAN_REVIEW_REQUIRED", "automatic_v2_created": False, "implementation_authorized": False, "synthetic": True,
        })
    for new in new_company_rows:
        review_queue.append({
            "review_id": f"REV9-{len(review_queue)+1:03d}", "review_type": "NEW_COMPANY_INTAKE",
            "recommendation_id": "NONE", "conglomerate_id": new["conglomerate_id"], "trigger": "UNIVERSE_GROWTH",
            "impact_records": 0, "priority": "MEDIUM", "status": "HUMAN_REVIEW_REQUIRED",
            "automatic_v2_created": False, "implementation_authorized": False, "synthetic": True,
        })
        industry = new["industry"]
        for rank, old in enumerate(recs[:3], 1):
            experience_rows.append({
                "experience_link_id": f"EXP9-{len(experience_rows)+1:03d}", "new_conglomerate_id": new["conglomerate_id"],
                "reference_recommendation_id": old["recommendation_id"], "similarity_rank": rank,
                "similarity_basis": f"Synthetic functional-pattern comparison for {industry}", "reuse_permission": "HYPOTHESIS_ONLY",
                "automatic_recommendation_allowed": False, "human_review_required": True, "synthetic": True,
            })
    write_csv(data / "tax_codes.csv", codes)
    write_csv(data / "tax_code_versions.csv", code_versions)
    write_csv(data / "quarterly_code_changes.csv", change_rows)
    write_csv(data / "conglomerates.csv", conglomerates)
    write_csv(data / "entities.csv", entities)
    write_csv(data / "ownership_edges.csv", ownership)
    write_csv(data / "intercompany_transactions.csv", txs)
    write_csv(data / "quarterly_impact_map.csv", impact_rows)
    write_csv(data / "recommendation_review_queue.csv", review_queue)
    write_csv(data / "experience_reuse_links.csv", experience_rows)
    report = f"""# Step 9 - Quarterly Tax-Code Intelligence and Universe Growth

## Quarterly event

Exactly ten of the 100 current tax-code modules changed for {NEXT_QUARTER}. The prior versions remain preserved, giving 110 rows in the version-history register while the current-code universe remains exactly 100 modules. Five synthetic companies joined the universe, each with four entities, three ownership edges, and five intercompany transactions.

## Code changes

{md_table(change_rows, [('change_id','Change'),('tax_code_id','Code'),('jurisdiction','Jurisdiction'),('module','Module'),('old_parameter','Old'),('new_parameter','New'),('delta','Delta'),('human_validated','Validated')])}

## Review queue

{md_table(review_queue, [('review_id','Review'),('review_type','Type'),('conglomerate_id','Group'),('recommendation_id','Recommendation'),('priority','Priority'),('status','Status'),('automatic_v2_created','Auto V2')])}

## Knowledge-base learning

Each new company is linked to three earlier recommendation records as hypothesis-only experience. Historical experience may suggest questions and candidate patterns; it cannot create a recommendation or override current law, facts, substance, or human review.

## Governance conclusion

No Recommendation V2 is generated automatically. Six existing recommendations and five new companies enter a human review queue. All historical recommendation versions remain intact.
"""
    write_text(package / "STEP_9_QUARTERLY_INTELLIGENCE_REPORT.md", report)
    write_text(package / "STEP_9_CHANGE_CONTROL_PLAYBOOK.md", "# Step 9 Quarterly Change-Control Playbook\n\n1. Freeze the prior quarter.\n2. Capture new source snapshots.\n3. Version only changed modules.\n4. Recompute the impact graph.\n5. Add new companies through controlled intake.\n6. Use experience only as hypothesis generation.\n7. Route affected recommendations to human review.\n8. Never overwrite recommendation history or create V2 automatically.\n")
    for name in ["quarterly_code_changes.csv", "tax_code_versions.csv", "quarterly_impact_map.csv", "recommendation_review_queue.csv", "experience_reuse_links.csv"]:
        shutil.copy2(data / name, package / f"STEP_9_{name.upper()}")
    write_text(vault / "11_Quarterly_Updates" / NEXT_QUARTER / "QUARTERLY_SUMMARY.md", report)
    write_text(vault / "13_Audit" / "STEP_9_SUCCESS.md", "# Step 9 Success\n\nExactly 10% of code modules changed; five synthetic companies were added; no Recommendation V2 was created automatically.")
    write_text(package / "STEP_9_README.md", "# Tax Planning Exo-Brain - Baby Step 9\n\nThe first quarterly cycle versions exactly 10 of 100 codes, adds five synthetic companies, maps impacts, and creates a human review queue without automatic Recommendation V2.")
    create_notebook(9, package, "Quarterly Tax-Code Intelligence and Universe Growth", "Version exactly 10% of tax-code modules, admit five synthetic companies, compute targeted impacts, and reuse accumulated experience only as human-reviewed hypotheses.", "100 current modules, 110 version-history rows, exactly 10 changes, 15 conglomerates, 174 entities, 159 ownership edges, 251 transactions, 11 review-queue items, no automatic V2, and validation PASS.")
    checks = {
        "prior_step_success": (vault / "13_Audit" / "STEP_8_SUCCESS.md").exists(),
        "current_codes_equal_100": len(codes) == 100,
        "changed_codes_equal_10": len(change_rows) == 10,
        "exactly_ten_percent_changed": len(change_rows) / len(codes) == 0.10,
        "version_history_equal_110": len(code_versions) == 110,
        "five_new_companies": len(new_company_rows) == 5 and len(conglomerates) == 15,
        "twenty_new_entities": len(new_entity_rows) == 20 and len(entities) == 174,
        "fifteen_new_ownership_edges": len(new_edge_rows) == 15 and len(ownership) == 159,
        "twenty_five_new_transactions": len(new_tx_rows) == 25 and len(txs) == 251,
        "review_queue_equal_11": len(review_queue) == 11,
        "experience_links_equal_15": len(experience_rows) == 15,
        "no_automatic_v2": not any(r["automatic_v2_created"] for r in review_queue),
        "historical_recommendations_preserved": len(recs) == 10 and all(r["version"] == "2026-Q3-R001" for r in recs),
    }
    finalize_step(9, package, {"checks": checks, "counts": {"current_tax_codes": 100, "tax_code_versions": 110, "changed_codes": 10, "conglomerates": 15, "entities": 174, "ownership_edges": 159, "transactions": 251, "impact_records": len(impact_rows), "review_queue": 11, "experience_links": 15}}, ["STEP_9_QUARTERLY_INTELLIGENCE_REPORT.md", "STEP_9_CHANGE_CONTROL_PLAYBOOK.md", "STEP_9_QUARTERLY_CODE_CHANGES.CSV", "STEP_9_RECOMMENDATION_REVIEW_QUEUE.CSV", "STEP_9_README.md"])


def build_step10() -> None:
    _, package, vault = copy_vault(10)
    data = vault / "16_Data"
    codes = read_csv(data / "tax_codes.csv")
    conglomerates = read_csv(data / "conglomerates.csv")
    recs = read_csv(data / "recommendations.csv")
    review_queue = read_csv(data / "recommendation_review_queue.csv")
    changes = read_csv(data / "quarterly_code_changes.csv")
    permissions = [
        {"role": "VIEWER", "read_vault": True, "export_bounded": False, "change_data": False, "approve": False, "transmit": False, "implement": False},
        {"role": "ANALYST", "read_vault": True, "export_bounded": True, "change_data": False, "approve": False, "transmit": False, "implement": False},
        {"role": "REVIEWER", "read_vault": True, "export_bounded": True, "change_data": False, "approve": False, "transmit": False, "implement": False},
        {"role": "COMMITTEE_OBSERVER", "read_vault": True, "export_bounded": True, "change_data": False, "approve": False, "transmit": False, "implement": False},
        {"role": "SYSTEM_AUDITOR", "read_vault": True, "export_bounded": True, "change_data": False, "approve": False, "transmit": False, "implement": False},
    ]
    write_csv(data / "application_permissions.csv", permissions)
    health_checks = [
        ("HC10-01", "Current tax-code count", len(codes) == 100, len(codes)),
        ("HC10-02", "Quarterly changes", len(changes) == 10, len(changes)),
        ("HC10-03", "Conglomerate count", len(conglomerates) == 15, len(conglomerates)),
        ("HC10-04", "Historical recommendations", len(recs) == 10, len(recs)),
        ("HC10-05", "Review queue", len(review_queue) == 11, len(review_queue)),
        ("HC10-06", "No automatic V2", not any(r["automatic_v2_created"].lower() == "true" for r in review_queue), 0),
        ("HC10-07", "No implementation authority", not any(r.get("implementation_authorized", "False").lower() == "true" for r in recs), 0),
        ("HC10-08", "Permissions are read-only", not any(r["change_data"] or r["approve"] or r["transmit"] or r["implement"] for r in permissions), 5),
        ("HC10-09", "Step 9 audit marker", (vault / "13_Audit" / "STEP_9_SUCCESS.md").exists(), 1),
        ("HC10-10", "Synthetic flag on codes", all(r["synthetic"].lower() == "true" for r in codes), len(codes)),
        ("HC10-11", "Synthetic flag on groups", all(r["synthetic"].lower() == "true" for r in conglomerates), len(conglomerates)),
        ("HC10-12", "Review items require humans", all(r["status"] == "HUMAN_REVIEW_REQUIRED" for r in review_queue), len(review_queue)),
    ]
    health_rows = [{"health_check_id": x[0], "control": x[1], "passed": x[2], "observed": x[3], "checked_at": BUILD_DATE, "synthetic": True} for x in health_checks]
    write_csv(data / "application_health_checks.csv", health_rows)
    scheduler_rows = [
        {"job_id": "JOB10-01", "job": "Freeze prior quarter", "sequence": 1, "cadence": "QUARTERLY", "mode": "READ_ONLY_PREVIEW", "human_trigger_required": True},
        {"job_id": "JOB10-02", "job": "Ingest synthetic source snapshots", "sequence": 2, "cadence": "QUARTERLY", "mode": "READ_ONLY_PREVIEW", "human_trigger_required": True},
        {"job_id": "JOB10-03", "job": "Detect candidate code changes", "sequence": 3, "cadence": "QUARTERLY", "mode": "READ_ONLY_PREVIEW", "human_trigger_required": True},
        {"job_id": "JOB10-04", "job": "Validate exact version changes", "sequence": 4, "cadence": "QUARTERLY", "mode": "READ_ONLY_PREVIEW", "human_trigger_required": True},
        {"job_id": "JOB10-05", "job": "Admit new-company intake records", "sequence": 5, "cadence": "QUARTERLY", "mode": "READ_ONLY_PREVIEW", "human_trigger_required": True},
        {"job_id": "JOB10-06", "job": "Compute affected subgraphs", "sequence": 6, "cadence": "QUARTERLY", "mode": "READ_ONLY_PREVIEW", "human_trigger_required": True},
        {"job_id": "JOB10-07", "job": "Generate human review queue", "sequence": 7, "cadence": "QUARTERLY", "mode": "READ_ONLY_PREVIEW", "human_trigger_required": True},
        {"job_id": "JOB10-08", "job": "Run integrity and governance checks", "sequence": 8, "cadence": "QUARTERLY", "mode": "READ_ONLY_PREVIEW", "human_trigger_required": True},
    ]
    write_csv(data / "quarterly_scheduler.csv", scheduler_rows)
    app_data = {
        "generated_at": BUILD_DATE,
        "quarter": NEXT_QUARTER,
        "synthetic": True,
        "governance": {"mode": "READ_ONLY", "recommendation_approval": False, "transmission": False, "implementation": False},
        "summary": {"jurisdictions": 20, "current_tax_codes": len(codes), "code_versions": csv_count(data, "tax_code_versions.csv"), "conglomerates": len(conglomerates), "entities": csv_count(data, "entities.csv"), "transactions": csv_count(data, "intercompany_transactions.csv"), "historical_recommendations": len(recs), "quarterly_changes": len(changes), "human_review_queue": len(review_queue)},
        "conglomerates": conglomerates,
        "recommendations": recs,
        "quarterly_changes": changes,
        "review_queue": review_queue,
        "health_checks": health_rows,
        "permissions": permissions,
        "scheduler": scheduler_rows,
    }
    app = vault / "15_Application" / "Read_Only_Operating_App"
    app.mkdir(parents=True, exist_ok=True)
    write_json(app / "data.json", app_data)
    css = """
:root{--ink:#102a43;--blue:#0072ce;--pale:#eaf2f8;--gold:#c8a951;--red:#a61b1b;--muted:#627d98;--bg:#f7fafc}*{box-sizing:border-box}body{margin:0;font-family:Inter,Arial,sans-serif;background:var(--bg);color:var(--ink)}header{background:linear-gradient(120deg,#0b2d4d,#155d88);color:white;padding:28px 5vw;border-bottom:5px solid var(--gold)}header h1{margin:0 0 6px;font-size:32px}header p{margin:0;opacity:.9}.bar{display:flex;gap:10px;flex-wrap:wrap;margin-top:14px}.badge{background:#ffffff1f;border:1px solid #ffffff55;border-radius:999px;padding:6px 11px;font-size:13px}main{padding:28px 5vw 50px}.cards{display:grid;grid-template-columns:repeat(auto-fit,minmax(155px,1fr));gap:14px}.card,.panel{background:white;border:1px solid #d9e2ec;border-radius:12px;box-shadow:0 4px 14px #102a4310}.card{padding:18px}.card .n{font-size:30px;font-weight:750;color:var(--blue)}.card .l{font-size:12px;text-transform:uppercase;letter-spacing:.06em;color:var(--muted)}.panel{margin-top:20px;padding:18px;overflow:auto}h2{font-size:20px;margin:0 0 14px}table{border-collapse:collapse;width:100%;font-size:13px}th,td{text-align:left;padding:9px 8px;border-bottom:1px solid #e6edf3;white-space:nowrap}th{color:var(--muted);font-size:11px;text-transform:uppercase}.pass{color:#087f5b;font-weight:700}.hold{color:var(--red);font-weight:700}.footer{margin-top:25px;padding:16px;border-left:4px solid var(--red);background:#fff4f4;font-size:13px}input{padding:9px 12px;border:1px solid #bcccdc;border-radius:8px;min-width:280px;margin-bottom:12px}
"""
    js = """
async function boot(){const d=await fetch('data.json').then(r=>r.json());const s=d.summary;document.querySelector('#cards').innerHTML=Object.entries(s).map(([k,v])=>`<div class="card"><div class="n">${v}</div><div class="l">${k.replaceAll('_',' ')}</div></div>`).join('');const hc=d.health_checks;document.querySelector('#health').innerHTML=hc.map(x=>`<tr><td>${x.health_check_id}</td><td>${x.control}</td><td class="${x.passed?'pass':'hold'}">${x.passed?'PASS':'FAIL'}</td><td>${x.observed}</td></tr>`).join('');window.groups=d.conglomerates;renderGroups(window.groups);document.querySelector('#changes').innerHTML=d.quarterly_changes.map(x=>`<tr><td>${x.tax_code_id}</td><td>${x.jurisdiction}</td><td>${x.module}</td><td>${x.old_parameter}</td><td>${x.new_parameter}</td><td>${x.delta}</td></tr>`).join('');document.querySelector('#queue').innerHTML=d.review_queue.map(x=>`<tr><td>${x.review_id}</td><td>${x.review_type}</td><td>${x.conglomerate_id}</td><td>${x.recommendation_id}</td><td>${x.priority}</td><td class="hold">${x.status}</td></tr>`).join('')}function renderGroups(rows){document.querySelector('#groups').innerHTML=rows.map(x=>`<tr><td>${x.conglomerate_id}</td><td>${x.name}</td><td>${x.industry}</td><td>${x.entity_count}</td><td>${x.recommendation_status}</td></tr>`).join('')}function filterGroups(q){q=q.toLowerCase();renderGroups(window.groups.filter(x=>Object.values(x).join(' ').toLowerCase().includes(q)))}boot();
"""
    index = f"""<!doctype html><html><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>Tax Planning Exo-Brain</title><link rel="stylesheet" href="styles.css"></head><body><header><h1>Tax Planning Exo-Brain</h1><p>Read-only synthetic operating application · {NEXT_QUARTER}</p><div class="bar"><span class="badge">READ ONLY</span><span class="badge">SYNTHETIC DATA</span><span class="badge">HUMAN REVIEW REQUIRED</span><span class="badge">NO IMPLEMENTATION AUTHORITY</span></div></header><main><section id="cards" class="cards"></section><section class="panel"><h2>System health</h2><table><thead><tr><th>ID</th><th>Control</th><th>Status</th><th>Observed</th></tr></thead><tbody id="health"></tbody></table></section><section class="panel"><h2>Company universe</h2><input placeholder="Filter companies" oninput="filterGroups(this.value)"><table><thead><tr><th>ID</th><th>Name</th><th>Industry</th><th>Entities</th><th>Status</th></tr></thead><tbody id="groups"></tbody></table></section><section class="panel"><h2>{NEXT_QUARTER} code changes</h2><table><thead><tr><th>Code</th><th>Jurisdiction</th><th>Module</th><th>Old</th><th>New</th><th>Delta</th></tr></thead><tbody id="changes"></tbody></table></section><section class="panel"><h2>Human review queue</h2><table><thead><tr><th>Review</th><th>Type</th><th>Group</th><th>Recommendation</th><th>Priority</th><th>Status</th></tr></thead><tbody id="queue"></tbody></table></section><div class="footer"><strong>Governance boundary:</strong> this prototype has no write, approve, transmit, instruct, file, or implement operation. It displays synthetic records only.</div></main><script src="app.js"></script></body></html>"""
    write_text(app / "styles.css", css)
    write_text(app / "app.js", js)
    write_text(app / "index.html", index)
    write_text(app / "README.md", "# Read-Only Operating Application\n\nServe this directory with `python -m http.server 8000`, then open `http://localhost:8000`. The application reads only `data.json`; it has no server, write endpoint, credentials, transmission capability, or implementation action.\n")
    playbooks = {
        "PB10_Quarterly_Cycle.md": "Freeze prior state; ingest sources; validate code changes; compute impacts; admit new companies; build review queue; run health checks; require human decisions.",
        "PB10_Recommendation_Review.md": "Open the impact set; inspect source and claim lineage; rerun relevant stress tests; document exceptions; draft but do not activate a new version; route to committee.",
        "PB10_Incident_Response.md": "Freeze outputs; preserve logs; identify affected objects; invalidate cache; re-run hashes and controls; escalate to system owner; document recovery.",
        "PB10_Data_Quality.md": "Check schema, uniqueness, referential integrity, effective dates, hashes, synthetic flags, and unresolved contradictions before reliance.",
        "PB10_Access_Control.md": "Grant least privilege; use read-only roles; prohibit credentials in vault; review access quarterly; log exports; revoke stale access.",
        "PB10_Model_Change.md": "Version assumptions and scoring; preserve prior outputs; benchmark; document impact; obtain human approval before any new model governs a recommendation.",
    }
    for name, body in playbooks.items():
        write_text(vault / "15_Application" / "Playbooks" / name, f"# {name[:-3].replace('_',' ')}\n\n{body}\n\nNo playbook creates authority to communicate or implement.")
    manifest = {
        "application": "Tax Planning Exo-Brain",
        "version": "10.0.0-synthetic",
        "mode": "READ_ONLY",
        "entrypoint": "15_Application/Read_Only_Operating_App/index.html",
        "data": "15_Application/Read_Only_Operating_App/data.json",
        "permissions": "16_Data/application_permissions.csv",
        "health_checks": "16_Data/application_health_checks.csv",
        "scheduler": "16_Data/quarterly_scheduler.csv",
        "playbooks": sorted(playbooks),
        "prohibited_capabilities": ["write", "approve", "transmit", "instruct", "file", "restructure", "implement"],
        "synthetic": True,
    }
    write_json(vault / "15_Application" / "application_manifest.json", manifest)
    architecture = """# Step 10 - Integrated Read-Only Operating Application

## Operating-system completion

The prototype integrates a dual-universe knowledge base, versioned tax-code intelligence, entity and transaction graphs, recommendation lineage, stress testing, evidence diligence, constrained optimization, adviser-process design, controlled instruction dry runs, quarterly change detection, and a read-only user interface.

## Application contract

The application is deliberately static. It reads a generated JSON snapshot and contains no server-side component, credentials, write endpoint, approval button, messaging channel, filing capability, or implementation action. Its five roles differ only in reading and bounded export permissions.

## Data state

- 20 synthetic jurisdictions.
- 100 current tax-code modules and 110 version-history records.
- 15 synthetic conglomerates, 174 entities, 159 ownership edges, and 251 intercompany transactions.
- 10 historical Recommendation V1 records; no automatic Recommendation V2.
- 10 quarterly code changes and 11 human-review queue items.

## Quarterly scheduler

Eight jobs describe the quarterly operating cadence. Each job is a read-only preview and requires a human trigger. The scheduler documents sequence and dependencies; it does not execute external actions.

## Health and permissions

Twelve health checks validate key counts, history preservation, human-review routing, read-only permissions, and the absence of implementation authority. The permission matrix denies change, approval, transmission, and implementation to every role.

## Conclusion

Step 10 completes a governed prototype, not a production tax engine. The most important product is not an optimized tax result but an operating architecture in which evidence, assumptions, contradictions, decisions, and permissions remain visible and contestable over time.
"""
    write_text(package / "STEP_10_OPERATING_SYSTEM_REPORT.md", architecture)
    write_text(package / "STEP_10_APPLICATION_MANUAL.md", "# Step 10 Application Manual\n\n## Launch\nUnzip the current vault, enter `15_Application/Read_Only_Operating_App`, run `python -m http.server 8000`, and open `http://localhost:8000`.\n\n## Views\nThe application shows system counts, health checks, the 15-company universe, 10 quarterly code changes, and the 11-item human review queue.\n\n## Security\nThe application is static and read-only. Do not add credentials or real taxpayer data. Any future write capability requires a separate threat model, access-control design, audit trail, and committee authorization.\n")
    shutil.copytree(app, package / "Read_Only_Operating_App")
    for name in ["application_permissions.csv", "application_health_checks.csv", "quarterly_scheduler.csv"]:
        shutil.copy2(data / name, package / f"STEP_10_{name.upper()}")
    shutil.copy2(vault / "15_Application" / "application_manifest.json", package / "STEP_10_APPLICATION_MANIFEST.json")
    write_text(vault / "13_Audit" / "STEP_10_SUCCESS.md", "# Step 10 Success\n\nIntegrated read-only prototype completed. All health checks pass; no production action is authorized.")
    write_text(package / "STEP_10_README.md", "# Tax Planning Exo-Brain - Baby Step 10\n\nThe integrated read-only prototype exposes the cumulative synthetic knowledge base, quarterly intelligence, review queue, health checks, permissions, playbooks, and scheduler without write or implementation capability.")
    create_notebook(10, package, "Integrated Read-Only Operating Application", "Assemble the complete governed prototype, static application, permissions, health checks, playbooks, and quarterly scheduler.", "15 conglomerates, 100 current codes, 110 code-version records, 12 passing health checks, five read-only roles, eight scheduled preview jobs, six playbooks, a static application, and validation PASS.")
    checks = {
        "prior_step_success": (vault / "13_Audit" / "STEP_9_SUCCESS.md").exists(),
        "app_files_present": all((app / name).exists() for name in ["index.html", "styles.css", "app.js", "data.json", "README.md"]),
        "health_checks_equal_12": len(health_rows) == 12,
        "all_health_checks_pass": all(r["passed"] for r in health_rows),
        "permissions_equal_5": len(permissions) == 5,
        "all_roles_read_only": not any(r["change_data"] or r["approve"] or r["transmit"] or r["implement"] for r in permissions),
        "scheduler_equal_8": len(scheduler_rows) == 8,
        "all_jobs_human_triggered": all(r["human_trigger_required"] for r in scheduler_rows),
        "playbooks_equal_6": len(playbooks) == 6,
        "no_automatic_v2": not any(r["automatic_v2_created"].lower() == "true" for r in review_queue),
        "implementation_never_authorized": not any(r.get("implementation_authorized", "False").lower() == "true" for r in recs),
    }
    finalize_step(10, package, {"checks": checks, "counts": app_data["summary"] | {"health_checks": 12, "roles": 5, "scheduler_jobs": 8, "playbooks": 6}}, ["STEP_10_OPERATING_SYSTEM_REPORT.md", "STEP_10_APPLICATION_MANUAL.md", "STEP_10_APPLICATION_MANIFEST.json", "STEP_10_APPLICATION_HEALTH_CHECKS.CSV", "STEP_10_README.md"])


def build_one(step: int) -> None:
    builders = {5: build_step5, 6: build_step6, 7: build_step7, 8: build_step8, 9: build_step9, 10: build_step10}
    if step not in builders:
        raise ValueError("Step must be 5 through 10")
    builders[step]()


def build_consolidated() -> None:
    target = ROOT / "Tax_Planning_ExoBrain_Steps_0_to_10_Deliverables"
    if target.exists():
        shutil.rmtree(target)
    for sub in ["01_Step_Packages", "02_Colab_Notebooks", "03_Current_Vault", "04_Principal_Reports", "05_Validation_and_Audit", "06_Read_Only_App"]:
        (target / sub).mkdir(parents=True, exist_ok=True)
    for step in range(11):
        step_zip = ROOT / f"Tax_Planning_ExoBrain_Step_{step}.zip"
        if not step_zip.exists():
            raise FileNotFoundError(step_zip)
        shutil.copy2(step_zip, target / "01_Step_Packages")
        package = ROOT / f"Tax_Planning_ExoBrain_Step_{step}"
        notebook = package / f"Baby_Step_{step}_Tax_Planning_ExoBrain.ipynb"
        if notebook.exists():
            shutil.copy2(notebook, target / "02_Colab_Notebooks")
        validation = package / f"STEP_{step}_VALIDATION.json"
        if validation.exists():
            shutil.copy2(validation, target / "05_Validation_and_Audit")
    shutil.copy2(ROOT / "Tax_Planning_ExoBrain_Current_Vault_Step_10.zip", target / "03_Current_Vault")
    for step in range(5, 11):
        package = ROOT / f"Tax_Planning_ExoBrain_Step_{step}"
        for file in package.iterdir():
            if file.is_file() and file.suffix.lower() in {".md", ".csv", ".json"} and ("REPORT" in file.name or "PLAYBOOK" in file.name or "MANUAL" in file.name or "README" in file.name):
                shutil.copy2(file, target / "04_Principal_Reports" / f"S{step}_{file.name}")
    shutil.copytree(ROOT / "Tax_Planning_ExoBrain_Step_10" / "Read_Only_Operating_App", target / "06_Read_Only_App" / "Read_Only_Operating_App")
    overview = """# Tax Planning Exo-Brain - Complete Steps 0-10

This consolidated package contains every cumulative step package, all eleven Colab notebooks, the final Step 10 Obsidian vault, principal reports, validation records, and the read-only operating application.

## Capability sequence

0. Dual-universe synthetic vault.
1. Recommendation V1 design loop.
2. Generalization and fragility testing.
3. Provenance, atomic claims, and contradictions.
4. Bounded tax-committee product.
5. Controlled diligence and confidence refresh.
6. Constrained multi-objective optimization.
7. Conflict-aware adviser-process design.
8. Recipient-controlled instruction and implementation dry runs.
9. Quarterly code intelligence, versioning, and universe growth.
10. Integrated read-only operating application.

All content is synthetic. No recommendation, filing, transmission, restructuring, appointment, or implementation is authorized.
"""
    write_text(target / "README_MASTER.md", overview)
    notebook_zip = ROOT / "Tax_Planning_ExoBrain_All_Colab_Notebooks_Steps_0_to_10.zip"
    zip_tree(target / "02_Colab_Notebooks", notebook_zip, "Tax_Planning_ExoBrain_All_Colab_Notebooks_Steps_0_to_10")
    shutil.copy2(notebook_zip, target)
    rows = []
    for file in sorted(p for p in target.rglob("*") if p.is_file() and p.name != "SHA256SUMS.txt"):
        rows.append(f"{sha256(file)}  {file.relative_to(target).as_posix()}")
    write_text(target / "SHA256SUMS.txt", "\n".join(rows))
    final_zip = ROOT / "Tax_Planning_ExoBrain_Steps_0_to_10_Complete_Deliverables.zip"
    zip_tree(target, final_zip, target.name)
    with zipfile.ZipFile(final_zip) as archive:
        bad = archive.testzip()
        if bad:
            raise AssertionError(bad)


def main() -> None:
    for step in range(5, 11):
        build_one(step)
    build_consolidated()






## Execute in Google Drive

The prior cumulative package is treated as immutable input. The result is written to a new step folder.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# The notebook builds the prior stages first in a temporary Colab workspace,
# then writes the requested cumulative vault and package to Drive.
ROOT = Path('/content/tax_planning_exobrain_build')
ROOT.mkdir(parents=True, exist_ok=True)
PROJECT = Path('/content/drive/MyDrive/Tax_Planning_ExoBrain_Project')

# Place the prior step package folder in ROOT, or change ROOT to the project
# directory that already contains Tax_Planning_ExoBrain_Step_5.
prior = PROJECT / 'Step_5' / 'Tax_Planning_ExoBrain_Step_5'
if prior.exists() and not (ROOT / prior.name).exists():
    shutil.copytree(prior, ROOT / prior.name)

build_one(6)
out = ROOT / 'Tax_Planning_ExoBrain_Step_6'
destination = PROJECT / 'Step_6'
if destination.exists():
    shutil.rmtree(destination)
shutil.copytree(out, destination / out.name)
print(json.loads((out / 'STEP_6_VALIDATION.json').read_text())['status'])
print(destination)


## Expected validated result

30 alternatives, 120 scenario tests, 10 selection records, nine internal design artifacts, one redesign block, zero approvals, and validation PASS.